# Timing from raw waveforms: student laboratory

This notebook does not hide the analysis inside one final fit. It runs the repository's beginner-facing producer and inspects the evidence in the order required for a physical timing claim.

The two teaching lanes are deliberate:

1. **Known physical pulses:** a clean synthetic sample injects per-stave jitter and should recover a B4--B6 residual near 0.1 ns.
2. **Known artifact:** a correct 8×18 frame is truncated and reinterpreted as 8×16. Pedestal boundaries also produce a central width near 0.1 ns, but with broad tails and no physical downstream pulses.

The same central number can therefore have opposite meanings. The earlier plots decide which interpretation is valid.

## 0. The four quantities

- **Sample interval:** distance between digitizer samples.
- **Timestamp:** interpolated crossing assigned to one waveform.
- **Pair residual:** corrected difference between two timestamps.
- **Single-stave resolution:** inferred detector parameter requiring a reference or validated multi-pair covariance model.

For a constant-fraction crossing,

\[
t_{\rm CFD}=t_k+\Delta t\frac{fA-y_k}{y_{k+1}-y_k}.
\]

This is continuous even when the sample interval is 10 ns.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

start = Path.cwd().resolve()
candidates = [start, *start.parents]
REPO = next(
    (candidate for candidate in candidates
     if (candidate / "chatgpt_todo/timing_supervisor_pack/student_timing_walkthrough.py").exists()),
    None,
)
if REPO is None:
    raise FileNotFoundError("Could not locate the ccb-testbeam repository root")
SCRIPT = REPO / "chatgpt_todo/timing_supervisor_pack/student_timing_walkthrough.py"
OUT = REPO / "reports/student_timing_walkthrough_notebook"
print("repository:", REPO)
print("script:", SCRIPT)
print("output:", OUT)

## 1. Run bounded known-answer tests

The self-test checks that the clean lane produces a sub-nanosecond pair and closes the multi-pair solution, while the artifact lane produces a narrow core with a much larger full RMS.

In [ ]:
completed = subprocess.run(
    [sys.executable, str(SCRIPT), "self-test"],
    check=True, capture_output=True, text=True,
)
print(completed.stdout.strip())

## 2. Generate the complete plot atlas

Ten thousand events are enough for the classroom demonstration and normally run in under a minute on a laptop. No Geant4 job is launched.

In [ ]:
completed = subprocess.run([
    sys.executable, str(SCRIPT), "demo",
    "--out", str(OUT),
    "--events", "10000",
    "--seed", "20260901",
], check=True, capture_output=True, text=True)
print("demo complete; report:", OUT / "STUDENT_REPORT.md")

In [ ]:
summary = json.loads((OUT / "analysis_summary.json").read_text())
summary.keys()

In [ ]:
def show_plot(filename, width=13):
    path = OUT / "plots" / filename
    image = Image.open(path)
    plt.figure(figsize=(width, width * image.height / image.width))
    plt.imshow(image)
    plt.axis("off")
    plt.show()

def pair_row(csv_name, fraction):
    table = pd.read_csv(OUT / csv_name)
    return table[(table.stave_a == "B4") & (table.stave_b == "B6") & (table.fraction == fraction)]

## 3. First prove the event frame

A clean source has one authorized word count. The comparison later shows how an apparently valid 8×16 reshape can still be the wrong source interpretation.

In [ ]:
show_plot("physical_01_word_count_contract.png")

## 4. Look at all channels

In the correct 8×18 lane, only B2 and its duplicate carry pulses. The legacy truncation turns pedestal boundaries into pulse-like steps in downstream blocks.

In [ ]:
show_plot("comparison_01_correct_vs_legacy_frame.png")

## 5. Check baseline and pulse identity

The component used for timing must be the component that passed the amplitude requirement, or it must be independently validated.

In [ ]:
show_plot("physical_05_component_identity.png")

## 6. Draw the CFD crossing

Each event shows the samples, selected peak, fraction threshold, bracket and interpolated crossing.

In [ ]:
show_plot("physical_06_cfd_examples_b2.png")

## 7. Count unique physical events

One two-stave event is not two events. Every rejection needs a named reason.

In [ ]:
show_plot("physical_07_cutflow.png")

## 8. Inspect each timestamp first

The pair difference is meaningful only after each timestamp and their correlation look physical.

In [ ]:
show_plot("physical_09_timestamp_correlation_b4_b6.png")

## 9. See the clean 0.1 ns residual

The synthetic physical lane has real pulses, known injected jitter, full RMS consistent with the central scale and multi-pair closure.

In [ ]:
show_plot("physical_12_residual_zoomed_core.png")

## 10. Scan the fraction

The fraction is a model choice. Select it on calibration/validation data, then evaluate the test set once.

In [ ]:
show_plot("physical_13_fraction_scan_core_vs_rms.png")

## 11. See the dangerous look-alike

The misframed pedestal steps also create a central width near 0.1 ns.

In [ ]:
show_plot("legacy_12_residual_zoomed_core.png")

## 12. Expose the artifact tails

The logarithmic view reveals a second scale: the central core is near 0.1 ns but the full RMS is about 4 ns.

In [ ]:
show_plot("legacy_11_residual_full_log.png")

## 13. Test multi-pair inference

A single B4--B6 width cannot determine B4 and B6 separately. Connected pairs and a covariance model are required.

In [ ]:
show_plot("physical_20_resolution_inference.png")

## 14. Require injection/recovery closure

The full method must recover known injected values over a grid, not only at one nominal point.

In [ ]:
show_plot("physical_22_injection_recovery_closure.png")

## 15. Compare the two B4--B6 numbers directly

The central values are intentionally similar. The full distribution and provenance are not.

In [ ]:
physical = pair_row("physical_subsample_timing_demo_pair_metrics.csv", 0.50)
artifact = pair_row("legacy_truncation_artifact_reproduction_pair_metrics.csv", 0.60)
pd.concat([
    physical.assign(lane="known physical pulses"),
    artifact.assign(lane="known framing artifact"),
], ignore_index=True)[[
    "lane", "fraction", "n", "sigma68_ns", "rms_ns",
    "core_sigma_ns", "chi2_ndf", "tail_gt2ns", "tail_gt10ns"
]]

## 16. Inspect recovered stave terms

The clean synthetic lane authorizes only **method closure**, not beam-data performance.

In [ ]:
pd.DataFrame({
    "injected_ns": summary["physical"]["inference"]["truth_sigma_ns"],
    "recovered_ns": summary["physical"]["inference"]["stave_sigma_ns"],
})

## 17. Move to raw data

Edit `student_timing_config.example.yaml` and keep the three claim gates false until their evidence is reviewed:

```yaml
source_frame_authorized: false
component_identity_authorized: false
allow_independent_zero_covariance_resolution_model: false
```

Then run:

```bash
python chatgpt_todo/timing_supervisor_pack/student_timing_walkthrough.py raw \
  --config chatgpt_todo/timing_supervisor_pack/student_timing_config.example.yaml \
  --out reports/student_timing_raw
```

The raw report should be read together with `STUDENT_RAW_TIMING_WALKTHROUGH.md` and `student_plot_atlas.csv`.

## Final oral-exam answer

A defensible answer is not “the histogram has sigma 0.1 ns.” It is:

> We validated the source frame and channel identity, showed event-level CFD crossings on physical pulses, froze calibration and fraction choices before the test runs, reported the full pair residual including RMS and tails, tested amplitude/slope/phase/run dependencies, measured a connected pair matrix, modeled covariance, and closed the stave-resolution inference on injected truth. Only then did we quote the authorized quantity and uncertainty.
